# Student Performance AI: Prediction & Analysis System

This notebook walks through the complete analytical story behind the
Student Performance AI project: from raw synthetic data to a trained,
evaluated classification model.

It is meant to be read alongside the modular `src/` package — every
step here calls the same functions used by `main.py` and `src/train.py`,
so the notebook and the production code never drift apart.

## Problem Statement

Can a student's academic **performance category** (`Low`, `Medium`,
`High`) be predicted from measurable study habits, attendance, prior
academic history, and a few contextual factors — *without* using their
actual final exam score as an input? This is framed as a supervised
multi-class classification problem.

## Objectives

- Load and validate a schema-checked dataset.
- Explore the data to understand distributions, relationships, and
  potential class imbalance.
- Engineer a leakage-safe preprocessing pipeline.
- Train and fairly compare several classification models.
- Evaluate the best model with classification-appropriate metrics.
- Demonstrate the trained model on example predictions.

## Dataset Description

**This is a synthetic dataset, not real student data.** It is generated
by `src/generate_dataset.py` using explicit, documented rules (see
`README.md`, section "Dataset", for the full explanation). In short:

- 1,200 synthetic students.
- Six numerical features (study hours, attendance, previous grades,
  sleep, socioeconomic index, midterm score).
- Four categorical features (parental support, extracurriculars,
  part-time job, internet access quality).
- A continuous `final_score`, generated as a weighted, noisy
  combination of the above.
- `performance_category`, derived directly from `final_score` using
  fixed thresholds (`<55`=Low, `<75`=Medium, else High).
- `student_id` and `final_score` are excluded from model features to
  prevent data leakage.

In [1]:
import sys
from pathlib import Path

# Allow importing the `src` package when running this notebook from
# the notebooks/ directory.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config
from src.data_loader import load_raw_dataset, DatasetNotFoundError
from src.preprocessing import split_features_target, build_preprocessing_pipeline
from src import visualization
from src.train import train_and_compare_models
from src.evaluate import summarize_metrics
from src.predict import predict_student_performance

config.ensure_directories()
pd.set_option("display.max_columns", None)

## Load Dataset

If the dataset doesn't exist yet, generate it first with `python -m src.generate_dataset`.

In [2]:
try:
    df = load_raw_dataset()
except DatasetNotFoundError as exc:
    print(exc)
    from src.generate_dataset import main as generate_dataset_main
    generate_dataset_main()
    df = load_raw_dataset()

print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
df.head()

Rows: 1200, Columns: 13


,student_id,study_hours_per_week,attendance_rate,previous_grade_avg,sleep_hours_per_night,parental_support_level,extracurricular_activities,part_time_job,internet_access_quality,socioeconomic_index,midterm_score,final_score,performance_category
0,1,16.83,66.13,61.76,8.06,Medium,Occasional,No,Good,50.51,44.27,77.04,High
1,2,8.76,76.17,81.72,9.94,High,Occasional,No,Good,60.85,62.84,71.94,Medium
2,3,19.50,87.04,71.94,4.99,Medium,Occasional,No,Good,49.71,52.22,71.00,Medium
3,4,20.64,80.77,58.23,8.70,High,NaN,No,Good,69.48,56.35,81.35,High
4,5,3.29,NaN,55.88,6.89,High,NaN,No,Poor,40.76,45.83,53.15,Low


## Data Validation

`load_raw_dataset()` already validated that every expected column is
present (via `data_loader.load_dataset`). Here we additionally check
dtypes, value ranges, and duplicate IDs.

In [3]:
print(df.dtypes)
print("\nDuplicate student_id count:", df["student_id"].duplicated().sum())
print("\nMissing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

student_id                      int64
study_hours_per_week          float64
attendance_rate               float64
previous_grade_avg            float64
sleep_hours_per_night         float64
parental_support_level            str
extracurricular_activities        str
part_time_job                     str
internet_access_quality           str
socioeconomic_index           float64
midterm_score                 float64
final_score                   float64
performance_category              str
dtype: object

Duplicate student_id count: 0

Missing values per column:
attendance_rate                24
sleep_hours_per_night          24
extracurricular_activities    353
internet_access_quality        24
dtype: int64


## Data Cleaning

Missing values (~2% in three columns, injected deliberately when the
synthetic data was generated) are **not** dropped here. They are
handled downstream by the preprocessing pipeline's imputers, so that
imputation statistics are fit only on training data and never leak
information from the test set. This cell simply confirms which rows
are affected.

In [4]:
rows_with_missing = df[df.isna().any(axis=1)]
print(f"Rows with at least one missing value: {len(rows_with_missing)} "
      f"({len(rows_with_missing) / len(df):.1%} of the dataset)")
rows_with_missing.head()

Rows with at least one missing value: 401 (33.4% of the dataset)


,student_id,study_hours_per_week,attendance_rate,previous_grade_avg,sleep_hours_per_night,parental_support_level,extracurricular_activities,part_time_job,internet_access_quality,socioeconomic_index,midterm_score,final_score,performance_category
3,4,20.64,80.77,58.23,8.70,High,NaN,No,Good,69.48,56.35,81.35,High
4,5,3.29,NaN,55.88,6.89,High,NaN,No,Poor,40.76,45.83,53.15,Low
5,6,7.19,73.91,44.94,7.85,High,NaN,No,Good,23.43,30.96,43.50,Low
6,7,15.77,73.45,72.07,4.66,High,NaN,No,Good,89.54,61.78,72.55,Medium
12,13,15.40,79.77,44.16,6.05,Low,NaN,No,Average,49.65,30.47,48.20,Low


## Exploratory Data Analysis

Each plot below is generated by `src/visualization.py` and saved to
`outputs/figures/`, then displayed inline here.

In [5]:
visualization.plot_target_distribution(df)
plt.show()

In [6]:
visualization.plot_numerical_distributions(df)
plt.show()

In [7]:
visualization.plot_correlation_heatmap(df)
plt.show()

In [8]:
visualization.plot_categorical_analysis(df)
plt.show()

**Reading the plots:** the target is imbalanced toward `Medium`
performers, which is why macro-averaged metrics (rather than plain
accuracy) are used for model selection later. The correlation heatmap
shows `midterm_score`, `previous_grade_avg`, and `study_hours_per_week`
as the numeric features most associated with `final_score` — expected,
since those three were weighted most heavily in the synthetic
generation process.

## Feature Engineering

No additional derived features are engineered beyond what the raw
dataset already provides — the existing behavioural and academic
fields are directly informative, and adding synthetic derived features
on top of an already-synthetic dataset would not add genuine signal.
The meaningful "engineering" work in this project is the **leakage-safe
preprocessing pipeline** (imputation, scaling, encoding), covered next.

## Train/Test Split

Handled inside `src/train.train_and_compare_models()` using
`sklearn.model_selection.train_test_split` with `stratify=y` (to keep
class proportions consistent across splits) and a fixed random seed
for reproducibility. Feature/target separation itself is shown below.

In [9]:
x, y = split_features_target(df)
print("Feature columns:", list(x.columns))
print("\nTarget distribution:")
print(y.value_counts())

Feature columns: ['study_hours_per_week', 'attendance_rate', 'previous_grade_avg', 'sleep_hours_per_night', 'parental_support_level', 'extracurricular_activities', 'part_time_job', 'internet_access_quality', 'socioeconomic_index', 'midterm_score']

Target distribution:
performance_category
Medium    749
Low       241
High      210
Name: count, dtype: int64


## Preprocessing

`build_preprocessing_pipeline()` returns an **unfitted**
`ColumnTransformer`:

- Numerical features → median imputation → standard scaling.
- Categorical features → most-frequent imputation → one-hot encoding.

It must be fit only on the training split; `train.py` does this inside
each model's own `Pipeline`, so preprocessing is refit independently
for every cross-validation fold as well — this is what prevents
information from the validation fold leaking into the imputer/scaler
statistics.

In [10]:
preprocessor = build_preprocessing_pipeline()
transformed_preview = preprocessor.fit_transform(x)
print("Transformed shape:", transformed_preview.shape)
print("Output feature names (first 10):", list(preprocessor.get_feature_names_out())[:10])

Transformed shape: (1200, 16)
Output feature names (first 10): ['numerical__study_hours_per_week', 'numerical__attendance_rate', 'numerical__previous_grade_avg', 'numerical__sleep_hours_per_night', 'numerical__socioeconomic_index', 'numerical__midterm_score', 'categorical__parental_support_level_High', 'categorical__parental_support_level_Low', 'categorical__parental_support_level_Medium', 'categorical__extracurricular_activities_Occasional']


## Model Training

Four models are trained and compared: Logistic Regression, Decision
Tree, Random Forest, and Gradient Boosting. Each is wrapped in a
`Pipeline` with its own preprocessing step and evaluated with 5-fold
stratified cross-validation (macro-F1) plus a held-out test set.

**Note:** running this cell retrains all four models from scratch and
will overwrite `models/best_model.joblib` and the metrics/figures in
`outputs/`. Numbers may vary slightly across environments due to
minor library-version differences, but the qualitative ranking should
be stable given the fixed random seed.

In [11]:
best_result, all_results = train_and_compare_models()

[logistic_regression] CV macro-F1: 0.7362 (+/- 0.0327) | Test macro-F1: 0.7368


[decision_tree] CV macro-F1: 0.5925 (+/- 0.0124) | Test macro-F1: 0.5871


[random_forest] CV macro-F1: 0.6697 (+/- 0.0178) | Test macro-F1: 0.6900


[gradient_boosting] CV macro-F1: 0.6808 (+/- 0.0227) | Test macro-F1: 0.6821

Best model selected: logistic_regression (test macro-F1 = 0.7368)


## Model Comparison

In [12]:
comparison_df = pd.DataFrame(
    {
        "model": [r.name for r in all_results],
        "cv_mean_f1_macro": [r.cv_mean_f1 for r in all_results],
        "cv_std_f1_macro": [r.cv_std_f1 for r in all_results],
        "test_f1_macro": [r.test_metrics["f1_macro"] for r in all_results],
        "test_accuracy": [r.test_metrics["accuracy"] for r in all_results],
    }
).sort_values("test_f1_macro", ascending=False)
comparison_df

,model,cv_mean_f1_macro,cv_std_f1_macro,test_f1_macro,test_accuracy
0,logistic_regression,0.736210,0.032695,0.736845,0.779167
2,random_forest,0.669723,0.017782,0.689986,0.762500
3,gradient_boosting,0.680814,0.022728,0.682148,0.741667
1,decision_tree,0.592543,0.012391,0.587125,0.687500


In [13]:
visualization.plot_model_comparison(
    {r.name: r.test_metrics["f1_macro"] for r in all_results},
    metric_name="Test Macro F1-Score",
)
plt.show()

## Model Evaluation

The best model is selected by **test-set macro-F1**, not training
accuracy, and confirmed against its cross-validation score to check it
isn't an artifact of one lucky split.

In [14]:
print(f"Best model: {best_result.name}")
print(summarize_metrics(best_result.test_metrics))

Best model: logistic_regression
Accuracy: 0.7792 | Precision (macro): 0.7804 | Recall (macro): 0.7142 | F1 (macro): 0.7368


## Best Model — Detailed Report

In [15]:
import json
print(json.dumps(best_result.test_metrics["classification_report"], indent=2))

{
  "Low": {
    "precision": 0.6470588235294118,
    "recall": 0.6875,
    "f1-score": 0.6666666666666666,
    "support": 48.0
  },
  "Medium": {
    "precision": 0.8012422360248447,
    "recall": 0.86,
    "f1-score": 0.8295819935691319,
    "support": 150.0
  },
  "High": {
    "precision": 0.8928571428571429,
    "recall": 0.5952380952380952,
    "f1-score": 0.7142857142857143,
    "support": 42.0
  },
  "accuracy": 0.7791666666666667,
  "macro avg": {
    "precision": 0.7803860674704665,
    "recall": 0.7142460317460317,
    "f1-score": 0.736844791507171,
    "support": 240.0
  },
  "weighted avg": {
    "precision": 0.7864381622214103,
    "recall": 0.7791666666666667,
    "f1-score": 0.7768220793140408,
    "support": 240.0
  }
}


In [16]:
x_train, x_test, y_train, y_test = None, None, None, None
from sklearn.model_selection import train_test_split
x, y = split_features_target(df)
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_SEED, stratify=y
)
y_pred = best_result.pipeline.predict(x_test)
visualization.plot_confusion_matrix(y_test, y_pred, labels=config.PERFORMANCE_CATEGORIES)
plt.show()

## Feature Analysis

Feature importances are only meaningful for tree-based models
(Decision Tree, Random Forest, Gradient Boosting) since Logistic
Regression exposes coefficients instead. This cell shows importances
**if** the selected best model supports them; otherwise it explains
why not.

In [17]:
fitted_model = best_result.pipeline.named_steps["model"]
if hasattr(fitted_model, "feature_importances_"):
    feature_names = best_result.pipeline.named_steps["preprocessing"].get_feature_names_out()
    importances = pd.Series(fitted_model.feature_importances_, index=feature_names)
    visualization.plot_feature_importance(importances)
    plt.show()
elif hasattr(fitted_model, "coef_"):
    feature_names = best_result.pipeline.named_steps["preprocessing"].get_feature_names_out()
    # For multi-class logistic regression, average the absolute coefficient
    # magnitude across classes as a simple importance proxy.
    coef_importance = pd.Series(np.abs(fitted_model.coef_).mean(axis=0), index=feature_names)
    visualization.plot_feature_importance(coef_importance, filename="feature_importance.png")
    plt.show()
    print("Best model is linear (Logistic Regression); shown values are mean "
          "absolute coefficient magnitude across classes, not tree feature importance.")
else:
    print(f"{best_result.name} does not expose feature importances or coefficients.")

Best model is linear (Logistic Regression); shown values are mean absolute coefficient magnitude across classes, not tree feature importance.


## Example Predictions

Using the saved best-model pipeline through the same
`src/predict.py` interface `main.py` uses.

In [18]:
example_students = [
    {
        "study_hours_per_week": 25, "attendance_rate": 95, "previous_grade_avg": 88,
        "sleep_hours_per_night": 7.5, "socioeconomic_index": 65, "midterm_score": 90,
        "parental_support_level": "High", "extracurricular_activities": "Regular",
        "part_time_job": "No", "internet_access_quality": "Good",
    },
    {
        "study_hours_per_week": 6, "attendance_rate": 55, "previous_grade_avg": 45,
        "sleep_hours_per_night": 5.0, "socioeconomic_index": 30, "midterm_score": 40,
        "parental_support_level": "Low", "extracurricular_activities": "None",
        "part_time_job": "Yes", "internet_access_quality": "Poor",
    },
]

for student in example_students:
    result = predict_student_performance(student)
    print(result)

{'predicted_category': 'High', 'class_probabilities': {'High': 0.9988266109706155, 'Low': 7.102545602300744e-10, 'Medium': 0.0011733883191298198}}
{'predicted_category': 'Low', 'class_probabilities': {'High': 4.234606130259463e-09, 'Low': 0.9989544931692397, 'Medium': 0.0010455025961541445}}


## Findings

- On this synthetic dataset, **Logistic Regression** achieved the best
  test macro-F1 (see the Model Comparison table above for exact
  figures — this notebook does not hardcode numbers so they always
  reflect the most recent run).
- The dataset's most numerically-correlated features with
  `final_score` are `midterm_score`, `previous_grade_avg`, and
  `study_hours_per_week`.
- The `Medium` performance category is the majority class, and models
  consistently perform best on it; the minority `Low`/`High` classes
  are harder to separate, visible in the confusion matrix above.

## Limitations

- Synthetic data: no finding here should be read as a real
  educational insight.
- Class imbalance affects minority-class recall.
- No hyperparameter tuning was performed.
- Feature "importance" for the synthetic generation weights is not
  empirically grounded.

## Future Improvements

- Swap in a real, properly licensed dataset.
- Add `GridSearchCV`/`RandomizedSearchCV` hyperparameter tuning.
- Add SHAP-based explainability for individual predictions.
- Track experiments with a tool such as MLflow.

## Conclusion

This notebook reproduces, end-to-end, the same pipeline shipped in
`src/`: a validated data load, leakage-aware preprocessing, a fair
comparison of four classification models, transparent evaluation, and
a reusable prediction interface — built on a clearly-labelled synthetic
dataset so every number in this notebook is honest about what it
does and doesn't represent.